# Task 2 — Clustering and image retrieval

Run all cells in order. Task 1 checkpoints and history CSV files must be available locally. Labels are used only for evaluation. The custom encoder was trained with labels in Task 1 and is frozen here.

In [ ]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score
from tqdm import tqdm
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE = Path.cwd()
if (BASE / 'notebooks').is_dir():
    BASE = BASE / 'notebooks'
TASK1 = BASE / 'task1_results'
if not TASK1.exists() and (BASE.parent / 'task1_results').exists():
    TASK1 = BASE.parent / 'task1_results'
ROOT = TASK1.parent / 'data'
OUTPUT = BASE / 'task2_results'
OUTPUT.mkdir(exist_ok=True)
BATCH_SIZE = 16
print('Device:', DEVICE)
print('Task 1 results:', TASK1)

In [ ]:
from sklearn.metrics.cluster import contingency_matrix

configs = {
    'baseline': (False, False, False),
    'batch_norm': (True, False, False),
    'dropout': (True, True, False),
    'augmentation': (True, True, False),
    'residual': (True, True, True),
}
selection = []
for name in configs:
    history_path = TASK1 / (name + '_history.csv')
    if not history_path.exists() or not (TASK1 / (name + '.pt')).exists():
        raise FileNotFoundError(f'Missing Task 1 history or checkpoint for {name}: {TASK1}')
    history = pd.read_csv(history_path)
    selection.append({'model': name, 'validation_f1': history.val_f1.max()})
selection = pd.DataFrame(selection)
best_custom = selection.loc[selection.validation_f1.idxmax(), 'model']
display(selection)
print('Selected custom encoder:', best_custom)

## Frozen feature extractors

Reuse the Task 1 model architecture and load its best validation checkpoint. ResNet uses original ImageNet weights. DINOv2 supplies self-supervised features.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels), nn.ReLU(), nn.Conv2d(channels, channels, 3, padding=1), nn.BatchNorm2d(channels))
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.layers(x))

class PetCNN(nn.Module):
    def __init__(self, batch_norm=False, dropout=False, residual=False, multitask=False):
        super().__init__()
        layers = []
        for incoming, outgoing in [(3, 32), (32, 64), (64, 128), (128, 256)]:
            layers.append(nn.Conv2d(incoming, outgoing, 3, padding=1))
            if batch_norm:
                layers.append(nn.BatchNorm2d(outgoing))
            layers.extend([nn.ReLU(), nn.MaxPool2d(2)])
        if residual:
            layers.append(ResidualBlock(256))
        self.features = nn.Sequential(*layers)
        self.embedding = nn.Sequential(nn.Flatten(), nn.Linear(256 * 14 * 14, 512), nn.ReLU(), nn.Dropout(0.5) if dropout else nn.Identity())
        self.breed_head = nn.Linear(512, 37)
        self.species_head = nn.Linear(512, 2) if multitask else None

    def forward(self, x):
        x = self.embedding(self.features(x))
        breed = self.breed_head(x)
        return (breed, self.species_head(x)) if self.species_head is not None else breed

In [ ]:
custom = PetCNN(*configs[best_custom])
custom.load_state_dict(torch.load(TASK1 / (best_custom + '.pt'), map_location='cpu', weights_only=True))
custom_encoder = nn.Sequential(custom.features, custom.embedding)
resnet_weights = models.ResNet50_Weights.DEFAULT
resnet = models.resnet50(weights=resnet_weights)
resnet.fc = nn.Identity()
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True)
dino_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
extractors = {
    'custom': (custom_encoder, transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])),
    'resnet50': (resnet, resnet_weights.transforms()),
    'dinov2': (dino, dino_transform),
}
for model, _ in extractors.values():
    model.eval()
    model.requires_grad_(False)

## Data and feature extraction

Cluster trainval images. Use trainval as the retrieval gallery and the official test split as queries. Keep image order fixed across extractors.

In [ ]:
gallery_data = datasets.OxfordIIITPet(ROOT, split='trainval', download=True)
query_data = datasets.OxfordIIITPet(ROOT, split='test', download=True)
class_names = gallery_data.classes

def extract_features(model, transform, split):
    data = datasets.OxfordIIITPet(ROOT, split=split, transform=transform)
    loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    features, labels = [], []
    model.to(DEVICE)
    with torch.no_grad():
        for images, target in tqdm(loader, desc=split):
            features.append(model(images.to(DEVICE)).cpu().numpy())
            labels.append(target.numpy())
    model.cpu()
    return np.concatenate(features), np.concatenate(labels)

features = {}
for name, (model, transform) in extractors.items():
    gallery, gallery_labels = extract_features(model, transform, 'trainval')
    queries, query_labels = extract_features(model, transform, 'test')
    features[name] = {'gallery': gallery, 'queries': queries}
    np.savez_compressed(OUTPUT / (name + '_features.npz'), gallery=gallery, queries=queries, gallery_labels=gallery_labels, query_labels=query_labels)
    print(name, 'gallery', gallery.shape, 'queries', queries.shape)
print('Gallery:', len(gallery_data), 'Queries:', len(query_data))

## Choose the number of clusters

Fit PCA on normalized gallery features for clustering efficiency. Select K by silhouette and inspect the K-means elbow. Use the same fixed sample for silhouette comparisons.

In [ ]:
K_VALUES = [2, 4, 8, 12, 20, 30, 37, 45, 60]
cluster_features, assignments, search_rows = {}, {}, []
for name, arrays in features.items():
    x = normalize(arrays['gallery'])
    pca = PCA(n_components=min(50, x.shape[1], len(x) - 1), random_state=SEED)
    x = pca.fit_transform(x)
    cluster_features[name] = x
    print(name, 'PCA explained variance:', pca.explained_variance_ratio_.sum())
    for algorithm in ['K-means', 'GMM']:
        for k in tqdm(K_VALUES, desc=name + ' ' + algorithm):
            if algorithm == 'K-means':
                model = KMeans(n_clusters=k, n_init=10, random_state=SEED)
            else:
                model = GaussianMixture(n_components=k, covariance_type='diag', n_init=2, max_iter=200, reg_covar=1e-5, random_state=SEED)
            labels = model.fit_predict(x)
            score = silhouette_score(x, labels, sample_size=min(1500, len(x)), random_state=SEED)
            assignments[(name, algorithm, k)] = labels
            search_rows.append({'extractor': name, 'algorithm': algorithm, 'k': k, 'silhouette': score,
                                'inertia': model.inertia_ if algorithm == 'K-means' else np.nan,
                                'converged': True if algorithm == 'K-means' else model.converged_})
search = pd.DataFrame(search_rows)
search.to_csv(OUTPUT / 'cluster_search.csv', index=False)
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
for row, name in enumerate(features):
    subset = search[search.extractor == name]
    km = subset[subset.algorithm == 'K-means']
    axes[row, 0].plot(km.k, km.inertia, 'o-')
    axes[row, 0].set(title=name + ' elbow', xlabel='K', ylabel='Inertia')
    for algorithm in ['K-means', 'GMM']:
        values = subset[subset.algorithm == algorithm]
        axes[row, 1].plot(values.k, values.silhouette, 'o-', label=algorithm)
    axes[row, 1].set(title=name + ' silhouette', xlabel='K', ylabel='Silhouette')
    axes[row, 1].legend()
plt.tight_layout()
plt.show()

## Evaluate all extractor and algorithm combinations

Choose among converged candidates using silhouette. External breed labels are introduced only for the final comparison.

In [ ]:
rows = []
for (name, algorithm), candidates in search.groupby(['extractor', 'algorithm'], sort=False):
    candidates = candidates[candidates.converged]
    if candidates.empty:
        raise RuntimeError(f'No converged candidates for {name}, {algorithm}; inspect the search output.')
    chosen = candidates.loc[candidates.silhouette.idxmax()]
    k = int(chosen.k)
    labels = assignments[(name, algorithm, k)]
    table = contingency_matrix(gallery_labels, labels)
    rows.append({'extractor': name, 'algorithm': algorithm, 'k': k, 'silhouette': chosen.silhouette,
                 'davies_bouldin': davies_bouldin_score(cluster_features[name], labels),
                 'ARI': adjusted_rand_score(gallery_labels, labels),
                 'NMI': normalized_mutual_info_score(gallery_labels, labels),
                 'purity': table.max(axis=0).sum() / table.sum()})
comparison = pd.DataFrame(rows)
comparison.to_csv(OUTPUT / 'cluster_comparison.csv', index=False)
display(comparison)
best = comparison.loc[comparison.silhouette.idxmax()]
print('Best internal score:', best.extractor, best.algorithm, 'K =', int(best.k))
print('Reference counts: 37 breeds and 2 species.')

## Visualize clusters

Show PCA and t-SNE projections. If the selected solution has fewer than four clusters, also show a four-cluster solution to inspect at least four groups.

In [ ]:
name, algorithm, k = best.extractor, best.algorithm, int(best.k)
x = cluster_features[name]
labels = assignments[(name, algorithm, k)]
projection_pca = PCA(n_components=2, random_state=SEED).fit_transform(x)
projection_tsne = TSNE(n_components=2, init='pca', learning_rate='auto', perplexity=30, random_state=SEED).fit_transform(x)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, projection, title in zip(axes, [projection_pca, projection_tsne], ['PCA', 't-SNE']):
    ax.scatter(projection[:, 0], projection[:, 1], c=labels, cmap='tab20', s=7, alpha=0.7)
    ax.set_title(f'{name}: {title}, K={k}')
plt.show()
grid_labels = labels
if len(np.unique(labels)) < 4:
    grid_labels = assignments[(name, algorithm, 4)]
    print('Additional four-cluster solution for image inspection.')
cluster_ids = pd.Series(grid_labels).value_counts().index[:4]
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
for row, cluster in enumerate(cluster_ids):
    indices = np.flatnonzero(grid_labels == cluster)
    counts = pd.Series(gallery_labels[indices]).value_counts().head(3)
    print('Cluster', cluster, 'size', len(indices), 'most common breeds:', [(class_names[i], int(count)) for i, count in counts.items()])
    sample = np.random.default_rng(SEED).choice(indices, size=min(5, len(indices)), replace=False)
    for ax in axes[row]:
        ax.axis('off')
    for ax, index in zip(axes[row], sample):
        image, target = gallery_data[int(index)]
        ax.imshow(image)
        ax.set_title(f'Cluster {cluster}\n{class_names[target]}', fontsize=8)
plt.tight_layout()
plt.show()

## Similar-image retrieval

Rank gallery images by cosine similarity. Average Precision@K across all test queries. Query and gallery splits are disjoint.

In [ ]:
def retrieve(query_feature, gallery_features, k=10):
    gallery = normalize(gallery_features)
    query = normalize(np.asarray(query_feature).reshape(1, -1))[0]
    scores = gallery @ query
    indices = np.argsort(-scores, kind='stable')[:k]
    return indices, scores[indices]

retrieval_rows = []
for name, arrays in features.items():
    gallery = normalize(arrays['gallery'])
    queries = normalize(arrays['queries'])
    hits = {k: [] for k in [1, 5, 10]}
    for start in range(0, len(queries), 128):
        scores = queries[start:start + 128] @ gallery.T
        indices = np.argsort(-scores, axis=1, kind='stable')[:, :10]
        relevant = gallery_labels[indices] == query_labels[start:start + len(indices), None]
        for k in hits:
            hits[k].extend(relevant[:, :k].mean(axis=1))
    retrieval_rows.append({'extractor': name, **{f'Precision@{k}': np.mean(values) for k, values in hits.items()}})
retrieval_table = pd.DataFrame(retrieval_rows)
retrieval_table.to_csv(OUTPUT / 'retrieval_metrics.csv', index=False)
display(retrieval_table)
query_indices = np.random.default_rng(SEED).choice(len(query_data), 3, replace=False)
for name, arrays in features.items():
    fig, axes = plt.subplots(3, 6, figsize=(16, 9))
    for row, index in enumerate(query_indices):
        image, target = query_data[int(index)]
        axes[row, 0].imshow(image)
        axes[row, 0].set_title('Query\n' + class_names[target], fontsize=8)
        neighbors, scores = retrieve(arrays['queries'][index], arrays['gallery'], k=5)
        for ax, neighbor, score in zip(axes[row, 1:], neighbors, scores):
            image, label = gallery_data[int(neighbor)]
            ax.imshow(image)
            ax.set_title(f'{class_names[label]}\n{score:.3f}', fontsize=8)
    for ax in axes.flat:
        ax.axis('off')
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

## Conclusion

After execution, compare the internal and external scores and retrieval precision. Explain whether the elbow supports the silhouette choice, whether K resembles two species or 37 breeds, and what visual properties appear in the sample clusters. Purity tends to grow with K. Discuss the supervised origin of the custom encoder and the effect of PCA preprocessing.